### Tarefa 1.1: Configuração básica e criação de um agente básico

Configure um protótipo de agente de atendimento ao cliente usando o framework do Strands Agents. Esse protótipo serve como ponto de partida para conhecer a jornada completa do protótipo do agente até as soluções prontas para produção.

Depois de concluir essa tarefa, seu agente terá a seguinte arquitetura básica:

<div style="text-align:left">
    <img src="images/architecture_lab1_strands_pt_br.png" width="75%"/>
</div>

*Descrição da imagem: protótipo de um agente simples executado localmente com ferramentas locais*

Instale dependências e importe todas as bibliotecas necessárias, incluindo o AWS SDK, os componentes do AgentCore e o framework do Strands para preparar o ambiente de desenvolvimento.

In [ ]:
import boto3
import json
import uuid
import time
import requests
from boto3.session import Session

# AgentCore imports
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore.memory.constants import StrategyType

# Strands imports
from strands import Agent
from strands.models import BedrockModel
from strands.tools.mcp import MCPClient
from strands.hooks import AfterInvocationEvent, HookProvider, HookRegistry, MessageAddedEvent

# Local tools
from lab_helpers.lab1_strands_agent import (
    get_product_info, get_return_policy, get_technical_support, web_search,
    SYSTEM_PROMPT, MODEL_ID
)
from lab_helpers.utils import get_ssm_parameter, put_ssm_parameter
from scripts.utils import get_cognito_client_secret

# Setup
boto_session = Session()
REGION = "us-east-1"
CUSTOMER_ID = "customer_001"
SESSION_ID = str(uuid.uuid4())

print("✅ Libraries imported successfully!")

Antes de criar o agente, analise as ferramentas locais que potencializarão nossos recursos de atendimento ao cliente. Abra e revise `lab_helpers/lab1_strands_agent.py` para entender o seguinte:

- As ferramentas são definidas localmente neste arquivo usando o decorador `@tool`
- As quatro funções da ferramenta e as respectivas finalidades:
  - get_product_info(): obtém informações sobre o produto
  - get_return_policy(): obtém a política de devolução para determinado produto
  - get_technical_support(): fornece orientações de suporte técnico
  - web_search(): pesquisa na web informações atualizadas
- Como elas usam dados simulados (simulando bancos de dados/APIs reais)
- O prompt do sistema que define o comportamento do agente

Crie um agente básico de suporte ao cliente que demonstre os principais recursos de IA, desde a compreensão das consultas até a execução de ações. Esse agente combina:

- **Modelo de base**: o “cérebro” que impulsiona o raciocínio e a tomada de decisões
- **Prompt do sistema**: instruções comportamentais que definem a personalidade e os padrões de serviço do agente
- **Ferramentas especializadas**: as quatro ferramentas locais (informações do produto, política de devolução, suporte técnico, pesquisa na web)

Quando você liga para o agente, ele segue este processo:
1. **Análise de consultas**: o agente analisa a pergunta do cliente
2. **Seleção de ferramentas**: o agente determina quais ferramentas usar (se houver)
3. **Execução da ferramenta**: o agente chama a ferramenta adequada com os parâmetros corretos
4. **Síntese da resposta**: o agente combina os resultados da ferramenta com o próprio conhecimento para criar uma resposta útil
5. **Verificação de qualidade**: o agente verifica se a resposta atende aos padrões no prompt do sistema

In [ ]:
# Create a basic agent with local tools
model = BedrockModel(model_id=MODEL_ID, temperature=0.3, region_name=REGION)
basic_agent = Agent(
    model=model,
    tools=[get_product_info, get_return_policy, get_technical_support, web_search],
    system_prompt=SYSTEM_PROMPT
)

print("✅ Basic customer support agent ready!")
print("📋 Available tools: Product Info, Return Policy, Technical Support, Web Search")

Teste seu agente básico para ver como ele lida com as consultas dos clientes e usa as ferramentas:

In [ ]:
# Test basic agent functionality
print("💬 Testing basic agent...\n")
response = basic_agent("What's the return policy for laptops?")
print("\n" + "="*50 + "\n")

Esse protótipo inicial tem várias limitações que você abordará nas próximas tarefas:

- **Sem memória persistente**: o agente esquece o histórico e as preferências do cliente das sessões anteriores
- **Somente ferramentas locais**: sem integração de ferramentas compartilhadas ou de nível corporativo  
- **Sem gerenciamento de identidade**: não é possível agir em nome de usuários específicos

### Tarefa 1.2: Aprimorar seu agente com memória

Um cliente valioso entra em contato com sua equipe de suporte sobre um problema com um pedido recente. O cliente explica as preferências, compartilha a frustração e trabalha com seu agente para resolver o problema. Três semanas depois, o cliente entra em contato com o suporte de novo com uma pergunta relacionada. Mas agora o cliente precisa repetir tudo, as preferências, o histórico, o contexto, porque seu agente só se lembra da sessão de conversa atual, e não das sessões anteriores. Isso cria:
- **Clientes frustrados** que precisam repetir as informações diversas vezes
- **Suporte ineficiente** que não se lembra de interações anteriores
- **Baixa satisfação do cliente** devido a respostas genéricas e impessoais

O Amazon Bedrock AgentCore Memory soluciona essa limitação fornecendo um serviço gerenciado que permite que os agentes de IA mantenham o contexto ao longo do tempo, lembrem-se de fatos importantes e ofereçam experiências consistentes e personalizadas. O AgentCore Memory opera em dois níveis:
- **Memória de curto prazo**: contexto imediato da conversa e informações baseadas na sessão (tratadas automaticamente pelo framework do Strands Agent)
- **Memória de longo prazo**: informações persistentes extraídas de várias conversas, incluindo fatos, preferências e resumos (implementadas por meio do serviço AgentCore Memory com estratégias USER_PREFERENCE e SEMANTIC)

Transforme seu protótipo em um assistente com reconhecimento do cliente que seja capaz de fazer o seguinte:
- **“Bem-vinda de volta, Sarah!”**: reconhece instantaneamente clientes recorrentes
- **“Acompanhamento da resolução do problema no laptop no mês passado”**: conecta perfeitamente as conversas relacionadas
- **“Com base no seu histórico de compras, recomendo o seguinte:”**: fornece sugestões personalizadas

Depois de concluir essa tarefa, o agente terá a seguinte arquitetura com recursos de memória integrados:

<div style="text-align:left">
    <img src="images/architecture_lab2_memory_pt_br.png" width="75%"/>
</div>

*Descrição da imagem: agente aprimorado com o AgentCore Memory para personalização e contexto persistentes do cliente*

**Configuração de estratégias de memória**: crie um recurso de memória que combina duas estratégias inteligentes:

| Tipo de estratégia | Objetivo | Benefício para o cliente |
|---------------|---------|------------------|
| USER_PREFERENCE | Preferências e comportamentos do cliente | “Lembro-me de que você prefere...” |
| SEMANTIC | Informações factuais e contexto | “Em relação ao seu problema anterior...” |

O AgentCore Memory usa namespaces para agrupar logicamente mensagens de memória de longo prazo usando o actorId:
- `support/customer/{actorId}/preferences`: para estratégia de memória de preferência do usuário
- `support/customer/{actorId}/semantic`: para estratégia de memória semântica

In [ ]:
# Initialize memory client for AgentCore Memory service
memory_client = MemoryClient(region_name=REGION)
memory_name = "CustomerSupportMemory"

def create_or_get_memory_resource():
    try:
        # Try to get existing memory resource from SSM parameter
        memory_id = get_ssm_parameter("/app/customersupport/agentcore/memory_id")
        memory_client.gmcp_client.get_memory(memoryId=memory_id)
        return memory_id
    except:
        # Create new memory resource with two strategies
        strategies = [
            {
                # USER_PREFERENCE strategy captures customer preferences and behaviors
                StrategyType.USER_PREFERENCE.value: {
                    "name": "CustomerPreferences",
                    "description": "Captures customer preferences and behavior",
                    "namespaces": ["support/customer/{actorId}/preferences"],
                }
            },
            {
                # SEMANTIC strategy stores factual information from conversations
                StrategyType.SEMANTIC.value: {
                    "name": "CustomerSupportSemantic",
                    "description": "Stores facts from conversations",
                    "namespaces": ["support/customer/{actorId}/semantic"],
                }
            },
        ]
        print("Creating AgentCore Memory resources (2-3 minutes)...")
        # Create memory resource and wait for completion
        response = memory_client.create_memory_and_wait(
            name=memory_name,
            description="Customer support agent memory",
            strategies=strategies,
            event_expiry_days=90,  # Memory events expire after 90 days
        )
        memory_id = response["id"]
        # Store memory ID in SSM for future use
        put_ssm_parameter("/app/customersupport/agentcore/memory_id", memory_id)
        return memory_id

memory_id = create_or_get_memory_resource()
print(f"✅ Memory resource ready: {memory_id}")

Simule um cliente recorrente chamado “customer_001” que teve interações anteriores com a equipe de suporte. Isso demonstra como o AgentCore Memory transforma automaticamente conversas individuais em informações ricas e persistentes do cliente. Carregue interações anteriores com clientes e observe o AgentCore Memory transformá-las automaticamente em informações de longo prazo sobre o cliente.

In [ ]:
# Seed previous customer interactions
previous_interactions = [
    ("I'm having issues with my MacBook Pro overheating during video editing.", "USER"),
    ("I can help with that thermal issue. Your MacBook Pro order #MB-78432 is still under warranty.", "ASSISTANT"),
    ("What's the return policy on gaming headphones? I need low latency for competitive FPS games", "USER"),
    ("For gaming headphones, you have 30 days to return. Since you're into competitive FPS, I'd recommend checking audio latency specs.", "ASSISTANT"),
    ("I need a laptop under $1200 for programming. Prefer 16GB RAM minimum and good Linux compatibility. I like ThinkPad models.", "USER"),
    ("Perfect! For development work, I'd suggest ThinkPad E series or Dell XPS models with excellent Linux support.", "ASSISTANT"),
]

if memory_id:
    memory_client.create_event(
        memory_id=memory_id,
        actor_id=CUSTOMER_ID,
        session_id="previous_session",
        messages=previous_interactions
    )
    print("✅ Customer history seeded successfully")
    print("⏳ Long-term memory processing will begin automatically...")

O Strands Agents fornece um avançado sistema de hooks que permite que os componentes reajam ou modifiquem o comportamento do agente por meio de retornos de chamada de eventos fortemente tipados. Ele garante que as operações de memória ocorram automaticamente sem intervenção manual.

Sempre que um cliente interage com seu agente, ele automaticamente:
- **Personaliza a conversa** com base em suas interações e preferências anteriores
- **Adiciona novas interações à memória** para melhorar continuamente a personalização futura

O que a integração de hooks faz:
- **Antes de responder**: recupera automaticamente o contexto e as preferências relevantes do cliente
- **Depois de responder**: salva automaticamente a nova interação no AgentCore Memory

Ative o contexto automático do cliente com os hooks de memória.

In [ ]:
class CustomerSupportMemoryHooks(HookProvider):
    def __init__(self, memory_id: str, client: MemoryClient, actor_id: str, session_id: str):
        self.memory_id = memory_id
        self.client = client
        self.actor_id = actor_id
        self.session_id = session_id
        self.namespaces = {
            i["type"]: i["namespaces"][0]
            for i in self.client.get_memory_strategies(self.memory_id)
        }

    def retrieve_customer_context(self, event: MessageAddedEvent):
        # Hook that runs before agent responds to retrieve customer context
        messages = event.agent.messages
        # Only process user messages (not tool results)
        if messages[-1]["role"] == "user" and "toolResult" not in messages[-1]["content"][0]:
            user_query = messages[-1]["content"][0]["text"]
            
            try:
                all_context = []
                # Retrieve memories from each strategy namespace, both USER_PREFERENCE and SEMANTIC
                for context_type, namespace in self.namespaces.items():
                    memories = self.client.retrieve_memories(
                        memory_id=self.memory_id,
                        namespace=namespace.format(actorId=self.actor_id),
                        query=user_query,
                        top_k=3,  # Get top 3 relevant memories
                    )
                    # Extract text content from memory objects
                    for memory in memories:
                        if isinstance(memory, dict):
                            content = memory.get("content", {})
                            if isinstance(content, dict):
                                text = content.get("text", "").strip()
                                if text:
                                    all_context.append(f"[{context_type.upper()}] {text}")
                
                # Prepend customer context to user message
                if all_context:
                    context_text = "\n".join(all_context)
                    original_text = messages[-1]["content"][0]["text"]
                    messages[-1]["content"][0]["text"] = f"Customer Context:\n{context_text}\n\n{original_text}"
            except Exception as e:
                print(f"Failed to retrieve customer context: {e}")

    def save_support_interaction(self, event: AfterInvocationEvent):
        # Hook that runs after agent responds to save interaction to memory
        try:
            messages = event.agent.messages
            # Only save if we have both user and assistant messages
            if len(messages) >= 2 and messages[-1]["role"] == "assistant":
                customer_query = None
                agent_response = None
                
                # Find the most recent user query and assistant response
                for msg in reversed(messages):
                    if msg["role"] == "assistant" and not agent_response:
                        agent_response = msg["content"][0]["text"]
                    elif msg["role"] == "user" and not customer_query and "toolResult" not in msg["content"][0]:
                        customer_query = msg["content"][0]["text"]
                        break
                
                # Save the interaction to AgentCore Memory
                if customer_query and agent_response:
                    self.client.create_event(
                        memory_id=self.memory_id,
                        actor_id=self.actor_id,
                        session_id=self.session_id,
                        messages=[(customer_query, "USER"), (agent_response, "ASSISTANT")],
                    )
        except Exception as e:
            print(f"Failed to save support interaction: {e}")

    def register_hooks(self, registry: HookRegistry) -> None:
        # Register both hooks with the agent's hook registry
        registry.add_callback(MessageAddedEvent, self.retrieve_customer_context)
        registry.add_callback(AfterInvocationEvent, self.save_support_interaction)

print("✅ Memory hooks defined - Automatic customer personalization enabled!")
print("🧠 Your agent will now remember customers and personalize every interaction")

Crie e teste seu agente com memória aprimorada para saber como ele recupera o contexto do cliente e personaliza as respostas:

In [ ]:
# Create memory-enhanced agent with hooks
memory_hooks = CustomerSupportMemoryHooks(memory_id, memory_client, CUSTOMER_ID, SESSION_ID)

memory_agent = Agent(
    model=model,
    tools=[get_product_info, get_return_policy, get_technical_support, web_search],
    hooks=[memory_hooks],
    system_prompt=SYSTEM_PROMPT
)

print("✅ Memory-enhanced agent created!")
print("🧠 Agent will automatically retrieve customer context and save interactions")

In [ ]:
# Wait for memory processing to complete
print("⏳ Waiting 90 seconds for memory processing to complete...")
time.sleep(90)

# Test memory recall
print("🧠 Testing memory-enhanced agent...\n")
response = memory_agent("What are my laptop preferences?")
print("\n" + "="*50 + "\n")

### Tarefa 1.3: Escalar com a integração de gateway e o AgentCore Identity

Com a memória pronta, concentre-se em ferramentas avançadas e amplie o impacto. Ótimos agentes precisam de ferramentas que aproveitem ao máximo APIs e dados proprietários e de terceiros, o que permite que esses agentes trabalhem para clientes internos e externos. No entanto, criar, proteger e escalar ferramentas de agentes é difícil, tornando-se um obstáculo significativo para os clientes saírem dos protótipos de agentes e conseguirem o valor comercial real dos agentes na produção. O AgentCore Gateway serve como uma camada de conectividade que permite que os agentes de IA descubram, autentiquem e invoquem ferramentas reais usando um endpoint unificado do **protocolo de contexto para modelos (MCP)**. Isso é crucial para empresas que gerenciam centenas de APIs, recursos e ferramentas.

Principais benefícios:
- **Solução de servidor MCP totalmente gerenciada** sem gerenciamento de infraestrutura
- **Integração de APIs** e funções do Lambda
- **Interface uniforme** em diversas ferramentas
- **Autenticação** e autorização seguras
- **Descoberta e seleção de ferramentas semânticas**

##### O que você aprenderá:

**Centralização e reutilização de ferramentas:**
- Migre a pesquisa na web da ferramenta local para o AgentCore Gateway centralizado
- Integre as funções corporativas do Lambda (verificação de garantia)
- Crie uma infraestrutura de ferramentas compartilhada que vários tipos de agentes possam acessar

**Segurança de nível corporativo:**
- Implemente autenticação baseada em JWT com integração com o Cognito
- Configure a autorização de entrada segura para acesso ao gateway
- Estabeleça controle de acesso baseado em identidade para o uso da ferramenta

Isso cria uma base dimensionável na qual as ferramentas são gerenciadas centralmente e reutilizáveis em vários tipos de agentes, eliminando a duplicação de código e simplificando a manutenção.

O AgentCore Identity também está envolvido nesse processo. Ele permite que agentes de IA acessem com segurança os recursos da AWS e ajudem a lidar com a autenticação de autores de chamadas recebidas trabalhando com o Amazon Cognito. Ele também permite que agentes de IA acessem com segurança ferramentas e serviços de terceiros usando autenticação de saída, embora esse recurso do Agentcore Identity não seja usado neste laboratório.

<div style="text-align:left">
    <img src="images/architecture_lab3_identity_pt_br.png" width="75%"/>
</div>

Depois de concluir essa tarefa, o agente terá a seguinte arquitetura com recursos de gateway integrados:

<div style="text-align:left">
    <img src="images/architecture_lab3_gateway_pt_br.png" width="75%"/>
</div>

*Descrição da imagem: agente aprimorado com o AgentCore Gateway para gerenciamento seguro e centralizado de ferramentas e integração corporativa*

Crie o AgentCore Gateway para expor uma função do Lambda como um endpoint compatível com MCP. Para validar os autores de chamada autorizados a invocar suas ferramentas, configure a **autenticação de entrada** usando a autorização OAuth, o padrão para servidores MCP.

### Entendendo a autenticação de gateway

O AgentCore Gateway usa o **OAuth 2.0 com tokens JWT** para proteger o acesso às suas ferramentas. Isso evita que aplicações não autorizadas invoquem suas funções do Lambda.

**Conceitos-chave:**

1. **Provedor de autenticação**: o Amazon Cognito gerencia identidades e emite tokens
2. **Credenciais do cliente**: seu agente usa client_id e client_secret (como nome de usuário/senha para aplicações)
3. **Tokens JWT**: tokens de curta duração que provam que o agente está autorizado
4. **Clientes permitidos**: uma lista de permissões de IDs de clientes que podem acessar o Gateway

**Como funciona:**
```
Agente → Cognito: “Aqui estão o client_id e o client_secret”
Cognito → Agente: “Aqui está seu token de acesso JWT”
Agente → Gateway: “Aqui está meu token”
Gateway → Cognito: “Esse token é válido e de um cliente permitido?”
Gateway → Agente: “Acesso concedido”
```

**Observação de segurança**: essas credenciais foram pré-criadas para você e salvas com segurança no Armazenamento de parâmetros do SSM. As credenciais nunca devem ser codificadas no seu código

In [ ]:
# Retrieve authentication configuration from SSM Parameter Store
# These values were created by the CloudFormation template

# Client ID: Identifies which application is making the request
machine_client_id = get_ssm_parameter("/app/customersupport/agentcore/machine_client_id")
print(f"Machine Client ID: {machine_client_id}")

# Discovery URL: Tells the Gateway where to find Cognito's OAuth configuration
# This URL provides metadata about token endpoints, supported scopes, etc.
cognito_discovery_url = get_ssm_parameter("/app/customersupport/agentcore/cognito_discovery_url")
print(f"Discovery URL: {cognito_discovery_url}")

# Configure JWT-based authentication for the Gateway
auth_config = {
    "customJWTAuthorizer": {
        # Only tokens from this client ID will be accepted
        "allowedClients": [machine_client_id],
        # Gateway will fetch OAuth metadata from this URL
        "discoveryUrl": cognito_discovery_url
    }
}

print("✅ Authentication configuration ready")

### Criar o AgentCore Gateway

O Gateway atua como um proxy seguro entre seu agente e a função do Lambda de back-end. Ele é como um gateway de API projetado especificamente para agentes de IA.

**O que está sendo criado:**
- **Infraestrutura de gateway**: o principal recurso de gateway
- **Protocolo MCP**: protocolo-padrão para comunicação de ferramentas
- **Autorização JWT**: usando a configuração de autenticação que acabou de ser criada
- **Perfil do IAM**: permissões para invocar a função do Lambda

**O que acontece em seguida:**
1. Crie o Gateway (esta célula)
2. Adicione o destino do Lambda com definições de ferramentas (próxima célula)
   - Uma função do Lambda gerencia várias ferramentas: `check_warranty_status` e `web_search`
   - O Gateway passa o nome da ferramenta para o Lambda, que encaminha para o manipulador apropriado
3. Conecte seu agente ao Gateway

**Observação**: o modelo do CloudFormation implantou uma função do Lambda (`CustomerSupportLambda`) que pode lidar com várias operações de ferramentas. Isso é mais eficiente do que implantar funções do Lambda separadas para cada ferramenta.

In [ ]:
class CreationFailedError(Exception):
    def __init__(self, message):
        self.message = message
        super().__init__(self.message)

gateway_client = boto3.client("bedrock-agentcore-control", region_name=REGION)
gateway_name = "customersupport-gw"

try:
    print(f"Creating gateway: {gateway_name}")
    create_response = gateway_client.create_gateway(
        name=gateway_name,
        roleArn=get_ssm_parameter("/app/customersupport/agentcore/gateway_iam_role"),
        protocolType="MCP",  # Model Context Protocol
        authorizerType="CUSTOM_JWT",  # Use JWT tokens for auth
        authorizerConfiguration=auth_config,  # Our auth config from above
        description="Customer Support AgentCore Gateway",
    )
    gateway_id = create_response["gatewayId"]
    gateway_url = create_response["gatewayUrl"]
    put_ssm_parameter("/app/customersupport/agentcore/gateway_id", gateway_id)

    # Wait for Gateway to be ready
    print("Waiting for Gateway to be ready...")
    while True:
        status = gateway_client.get_gateway(gatewayIdentifier=gateway_id)['status']
        if status == 'READY':
            break
        elif status == 'FAILED':
            raise CreationFailedError("Gateway creation failed")
        else:
            print(f"  Status: {status}")
            time.sleep(5)

    print(f"✅ Gateway created successfully!")
    print(f"   Gateway ID: {gateway_id}")
    print(f"   Gateway URL: {gateway_url}")
    
except gateway_client.exceptions.ConflictException:
    # Gateway already exists, retrieve it
    gateway_id = get_ssm_parameter("/app/customersupport/agentcore/gateway_id")
    gateway_response = gateway_client.get_gateway(gatewayIdentifier=gateway_id)
    gateway_url = gateway_response["gatewayUrl"]
    print(f"✅ Using existing gateway: {gateway_id}")
    
except CreationFailedError:
    print("\033[31m❌ Gateway creation failed. Check CloudWatch logs for details.\033[0m")

O AgentCore Gateway preenche o contexto do Lambda com o nome da ferramenta a ser invocada, enquanto os parâmetros passados para a ferramenta são fornecidos no evento do Lambda. Isso permite que você integre as funções corporativas do Lambda (nesse caso, `AgentCoreLab-CustomerSupportLambda`) que podem ser reutilizadas em vários agentes.

Adicione suas funções do Lambda como destinos de gateway usando a especificação da API:

In [ ]:
# Load API specification for Lambda tools
api_spec = [
    {
        "name": "check_warranty_status",
        "description": "Check warranty status using serial number and email",
        "inputSchema": {
            "type": "object",
            "properties": {
                "serial_number": {"type": "string"},
                "customer_email": {"type": "string"}
            },
            "required": ["serial_number"]
        }
    },
    {
        "name": "web_search",
        "description": "Search the web for updated information",
        "inputSchema": {
            "type": "object",
            "properties": {
                "keywords": {"type": "string", "description": "Search query keywords"},
                "region": {"type": "string", "description": "Search region (e.g., us-en)"},
                "max_results": {"type": "integer", "description": "Maximum results"}
            },
            "required": ["keywords"]
        }
    }
]

# Create gateway target
lambda_target_config = {
    "mcp": {
        "lambda": {
            "lambdaArn": get_ssm_parameter("/app/customersupport/agentcore/lambda_arn"),
            "toolSchema": {"inlinePayload": api_spec},
        }
    }
}

try:
    create_target_response = gateway_client.create_gateway_target(
        gatewayIdentifier=gateway_id,
        name="LambdaTarget",
        description="Lambda tools for customer support",
        targetConfiguration=lambda_target_config,
        credentialProviderConfigurations=[{"credentialProviderType": "GATEWAY_IAM_ROLE"}],
    )
    print(f"✅ Gateway target created: {create_target_response['targetId']}")
except Exception as e:
    print(f"Gateway target may already exist: {str(e)}")

Integre o token de autenticação do Cognito em um MCPClient do Strands SDK para criar uma conexão MCP segura.

Crie um cliente MCP autenticado para acessar as ferramentas de gateway:

In [ ]:
def get_cognito_client_secret():
    # Get Cognito client secret using Cognito API
    client = boto3.client("cognito-idp")
    response = client.describe_user_pool_client(
        UserPoolId=get_ssm_parameter("/app/customersupport/agentcore/userpool_id"),
        ClientId=get_ssm_parameter("/app/customersupport/agentcore/machine_client_id"),
    )
    return response["UserPoolClient"]["ClientSecret"]

def get_oauth_token():
    # Get OAuth token for gateway authentication using client credentials flow
    headers = {"Content-Type": "application/x-www-form-urlencoded"}
    data = {
        "grant_type": "client_credentials",  # OAuth 2.0 client credentials flow
        "client_id": get_ssm_parameter("/app/customersupport/agentcore/machine_client_id"),
        "client_secret": get_cognito_client_secret(),
        "scope": get_ssm_parameter("/app/customersupport/agentcore/cognito_auth_scope"),
    }
    # Request access token from Cognito
    response = requests.post(
        get_ssm_parameter("/app/customersupport/agentcore/cognito_token_url"),
        headers=headers, data=data
    )
    return response.json()

# Get OAuth access token (JWT format)
token_response = get_oauth_token()
access_token = token_response['access_token']

# Create MCP client with Bearer token authentication
mcp_client = MCPClient(
    url=gateway_url,
    headers={"Authorization": f"Bearer {access_token}"},  # JWT token in Authorization header
)

print(f"✅ MCP client configured for gateway: {gateway_url}")

Combine tudo: hooks de memória, ferramentas locais e ferramentas de gateway. Isso cria uma arquitetura híbrida em que algumas ferramentas permanecem locais (para velocidade e simplicidade), enquanto outras são centralizadas no gateway (para reutilização e integração corporativa).

Essa abordagem elimina a duplicação de código entre diferentes agentes e cria um gerenciamento centralizado para atualizações de ferramentas:

In [ ]:
# Initialize memory hooks for customer context
memory_hooks = CustomerSupportMemoryHooks(memory_id, memory_client, CUSTOMER_ID, SESSION_ID)

# Start MCP client connection to gateway
mcp_client.start()
# Retrieve available tools from the gateway
gateway_tools = mcp_client.list_tools_sync()

# Combine local tools with centralized gateway tools
all_tools = [
    get_product_info,      # Local tool
    get_return_policy,     # Local tool
    get_technical_support, # Local tool
] + gateway_tools          # Gateway tools - web_search, check_warranty_status

# Create enhanced agent with memory and gateway integration
enhanced_agent = Agent(
    model=model,
    tools=all_tools,           # Local + gateway tools
    hooks=[memory_hooks],      # Automatic memory operations
    system_prompt=SYSTEM_PROMPT
)

print("✅ Enhanced Customer Support Agent created!")
print(f"📊 Total tools available: {len(all_tools)}")
print(f"🧠 Memory enabled with ID: {memory_id}")
print(f"🔒 Secure gateway integration: {gateway_url}")

Teste seu agente com recursos de memória e gateway. Verifique se o agente pode usar perfeitamente as ferramentas locais e as ferramentas de gateway centralizado, mantendo o contexto do cliente com a memória.

Os cenários de teste incluem verificação de garantia, pesquisa na web e recursos combinados de memória e gateway:

In [ ]:
# Test gateway tools
print("🔍 Testing gateway web search...\n")
response2 = enhanced_agent("Search for latest iPhone 15 troubleshooting tips")
print("\n" + "="*50 + "\n")

In [ ]:
# Test warranty check
print("🛡️ Testing warranty check...\n")
response3 = enhanced_agent("Check warranty status for serial number ABC12345678")
print("\n" + "="*50 + "\n")

In [ ]:
# Test combined capabilities
print("🎯 Testing combined memory + gateway capabilities...\n")
response4 = enhanced_agent("I need gaming headphones again, and also search for the latest reviews")
print("\n" + "="*50 + "\n")

## Próximas etapas

🎉 **Parabéns!** Você concluiu os exercícios do caderno.

Você concluiu as seguintes tarefas:
- Criou um protótipo básico de agente de IA usando o Strands
- Aprimorou o protótipo com o AgentCore Memory para um contexto persistente do cliente
- Integrou o AgentCore Gateway para compartilhamento seguro e centralizado de ferramentas
- Testou o sistema completo de suporte ao cliente pronto para produção

### Qual é o próximo passo?

1. **Feche este arquivo do caderno**
2. **Retorne às instruções do laboratório**
3. **Continue com a Tarefa 2** para conhecer o painel do AgentCore e ver seus recursos em ação
